# Short term financing and sensitivity analysis
*Algorithms for Optimization, Master Program in Finance, University of Pavia. Notebook of Lecture 3.*

| Section | Topic |
|---|---|
| 0 | Setup |
| 1 | The problem |
| 2 | Modeling |
| 3 | Solving |
| 4 | Features of linear programs |
| 5 | The sensitivity report |
| 6 | Answering questions with shadow prices |
| 7 | Sensitivity in portfolio allocation |

Exercises are marked with ✏️.

## 0. Setup

In [ ]:
!pip install gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/AmbrogioMB/AlgOpt/main/data/"

## 1. The problem
Corporations routinely face the problem of financing short term cash commitments. Linear programming can help in figuring out an optimal combination of financial instruments to meet these commitments.

A company has the following net cash flow requirements, in thousands of euros (negative = the company needs cash):

In [ ]:
cash = pd.read_csv(DATA_URL + "cash_flows.csv")
cash

The company has the following sources of funds:
* a **line of credit** of up to €100K at an interest rate of 1% per month;
* in any one of the first three months, it can issue 90-day **commercial paper** bearing a total interest of 2% for the 3-month period;
* **excess funds** can be invested at an interest rate of 0.3% per month.

Goal: maximize the company's wealth at the end of June.

## 2. Modeling
**Decision variables.**
* $x_i$: balance of the line of credit in month $i$ (January to May);
* $y_i$: amount of commercial paper issued in month $i$ (January to March);
* $z_i$: excess funds in month $i$ (January to May);
* $v$: the company's wealth in June.

**Objective.** $\max v$.

**Constraints.** For each month, *cash inflow = cash outflow*. In January the company needs 150, it can draw from the line of credit and issue commercial paper, and it may keep some excess funds:
$$x_1 + y_1 - z_1 = 150.$$
In February it needs 100, it repays the credit of January with interest ($1.01 x_1$) and receives the excess funds of January with interest ($1.003 z_1$):
$$x_2 + y_2 - 1.01 x_1 + 1.003 z_1 - z_2 = 100.$$
From April on, the commercial paper issued three months before has to be repaid ($1.02 y_{i-3}$), and new paper cannot be issued. In June the remaining cash is the wealth $v$.

In general, for month $i$ with net cash flow $f_i$:
$$x_i + y_i - 1.01 x_{i-1} - 1.02 y_{i-3} + 1.003 z_{i-1} - z_i = -f_i,$$
where the variables that do not exist are omitted, and $z_6$ is replaced by $v$. Finally $x_i \leq 100$ (credit limit) and $x_i, y_i, z_i \geq 0$.

We write the six balance constraints with a loop, and keep them in a dictionary to query them later.

In [ ]:
months = list(cash["month"])
flow = dict(zip(cash["month"], cash["net_cash_flow"]))

stf = gp.Model("Short term financing")
x = stf.addVars(months[:-1], name="x")   # line of credit, Jan-May
y = stf.addVars(months[:3], name="y")    # commercial paper, Jan-Mar
z = stf.addVars(months[:-1], name="z")   # excess funds, Jan-May
v = stf.addVar(lb=-GRB.INFINITY, name="v")  # wealth in June (could be negative)

stf.setObjective(v, GRB.MAXIMIZE)

balance = {}
for k, m in enumerate(months):
    inflow = gp.LinExpr()
    if m in x:
        inflow += x[m]
    if m in y:
        inflow += y[m]
    if k >= 1:
        inflow += -1.01 * x[months[k - 1]] + 1.003 * z[months[k - 1]]
    if k >= 3:
        inflow += -1.02 * y[months[k - 3]]
    inflow += -z[m] if m in z else -v
    balance[m] = stf.addConstr(inflow == -flow[m], name=f"balance_{m}")

credit = {m: stf.addConstr(x[m] <= 100, name=f"credit_{m}") for m in months[:-1]}

Always **check the model you wrote**. Gurobi can write it to a file in a readable format (`.lp`): compare it with the constraints above.

In [ ]:
stf.write("stf.lp")
print(open("stf.lp").read())

## 3. Solving
Setting the parameter `Method` to 0 forces Gurobi to use the primal simplex method (Lecture 2).

In [ ]:
stf.Params.Method = 0
stf.optimize()

if stf.Status == GRB.OPTIMAL:
    plan = pd.DataFrame({
        "credit x": [x[m].X if m in x else None for m in months],
        "paper y": [y[m].X if m in y else None for m in months],
        "excess z": [z[m].X if m in z else None for m in months],
    }, index=months)
    print(plan.round(3))
    print(f"Wealth in June: {v.X:.3f} thousand euros")

### Exercise 3.1 ✏️ (Exercise 2 of Lecture 3)
How would the formulation change if the commercial paper had a **2 month** maturity (still 2% total interest) instead of 3? Now paper can be issued from January to April. Write the new model and solve it. Is the company better off?

*Hint:* the cleanest way is to write a function `build_stf(maturity)` that returns the model, using the loop above with `months[k - maturity]`.

In [ ]:
# YOUR CODE HERE

## 4. Features of linear programs
Hidden in each LP formulation are a number of assumptions:
* **Proportionality**: the contribution of a variable is proportional to its value (no fixed fees, no volume discounts);
* **Additivity**: the contribution of a variable does not depend on the other variables;
* **Divisibility**: any fraction of a variable is allowed (otherwise: integer programming, Lectures 4 and 5);
* **Certainty**: all data are known (otherwise: stochastic programming, robust optimization).

It is very rare that a problem meets all of them exactly, but a model can still give useful managerial insight. Sensitivity analysis tells us how much the answer depends on the data.

## 5. The sensitivity report
For every constraint Gurobi gives the **shadow price** (`Pi`) and the range of right-hand sides in which it is valid (`SARHSLow`, `SARHSUp`). For every variable it gives the **reduced cost** (`RC`) and the range of objective coefficients in which the current solution stays optimal (`SAObjLow`, `SAObjUp`). We collect them in two DataFrames, like the Sensitivity Report of the Excel Solver.

In [ ]:
def sensitivity_report(model):
    variables = pd.DataFrame(
        [[v.VarName, v.X, v.RC, v.Obj, v.SAObjLow, v.SAObjUp] for v in model.getVars()],
        columns=["variable", "value", "reduced cost", "obj coeff", "obj low", "obj up"],
    ).set_index("variable")
    constraints = pd.DataFrame(
        [[c.ConstrName, c.RHS - c.Slack, c.Pi, c.RHS, c.SARHSLow, c.SARHSUp] for c in model.getConstrs()],
        columns=["constraint", "lhs", "shadow price", "rhs", "rhs low", "rhs up"],
    ).set_index("constraint")
    return variables, constraints


var_report, con_report = sensitivity_report(stf)
print(var_report.round(4))
print()
print(con_report.round(4))

**How to read the constraint report.** The shadow price $u$ of a constraint means:

> If the right-hand side of the constraint changes by $\Delta$, the optimal value changes by $u \Delta$, as long as the new right-hand side stays between `rhs low` and `rhs up`.

Outside this range the set of binding constraints changes, and we have to solve the LP again. Non-binding constraints (e.g., the credit limit in the months in which the credit line is not fully used) have shadow price zero.

## 6. Answering questions with shadow prices
Recall the balance constraint of month $i$: its right-hand side is $-f_i$, i.e., **the cash the company needs** in month $i$.

**(1) If the net cash flow in January were −200 (instead of −150), by how much would the wealth in June decrease?**
The right-hand side of `balance_Jan` goes from 150 to 200, $\Delta = 50$, within the allowable range. The shadow price is $-1.0373$, so the wealth changes by $-1.0373 \cdot 50 \approx -51.86$: it decreases by about €51,860.

Let us check it by solving the LP again.

In [ ]:
v_base, pi_jan = stf.ObjVal, balance["Jan"].Pi
print(f"shadow price {pi_jan:.4f}, allowable range of the rhs: [{balance['Jan'].SARHSLow:.2f}, {balance['Jan'].SARHSUp}]")
balance["Jan"].RHS = 200
stf.Params.OutputFlag = 0
stf.optimize()
print(f"new wealth {stf.ObjVal:.3f}, change {stf.ObjVal - v_base:.3f}, predicted {pi_jan * 50:.3f}")
balance["Jan"].RHS = 150  # restore the original data
stf.optimize()

**(2) If the net cash flow in March were 250 (instead of 200), by how much would the wealth in June increase?** See Exercise 6.1.

**(3) If the credit limit were increased from 100 to 200, would the wealth increase?**
The credit limit constraints have shadow price 0 in every month (the line of credit is never fully used), and the allowable increase is infinite: no effect on the wealth.

**(4) Part of the January requirement is due to the purchase of a machine worth €100,000. The vendor allows the payment to be made in June, at an interest rate of 3% for the 5-month period. Should the company accept? What if the rate were 4%?**
Paying in June means: the January requirement decreases by 100 ($\Delta = -100$, within the allowable decrease), and the wealth in June decreases by the payment, $100 \cdot 1.03 = 103$. With the shadow price $-1.0373$, reducing the January requirement by 100 increases the wealth by $103.73$. Since $103.73 > 103$ the company should accept. The break-even rate is $3.73\%$: at 4% ($104 > 103.73$) it should refuse.

### Exercise 6.1 ✏️
Answer question (2) with the shadow price of `balance_Mar`, then check your answer by changing the right-hand side and solving the LP again (do not forget to restore the original data).

In [ ]:
# YOUR CODE HERE

## 7. Sensitivity in portfolio allocation
Back to the wealth allocation problem of the Linear programming notebook (Section 3): maximize the expected return with at most 20% in each stock and at most 30% in Tech and in Banks.

In [ ]:
assets = pd.read_csv(DATA_URL + "assets.csv")
tickers = list(assets["ticker"])
mu = dict(zip(assets["ticker"], assets["exp_return"]))
u = dict(zip(assets["ticker"], assets["max_weight"]))
sector = dict(zip(assets["ticker"], assets["sector"]))

alloc = gp.Model("Wealth allocation")
alloc.Params.OutputFlag = 0
w = alloc.addVars(tickers, lb=0, ub=u, name="w")
alloc.setObjective(gp.quicksum(mu[i] * w[i] for i in tickers), GRB.MAXIMIZE)
alloc.addConstr(w.sum() == 1, name="budget")
limit = {s: alloc.addConstr(gp.quicksum(w[i] for i in tickers if sector[i] == s) <= 0.30, name=f"sector_{s}")
         for s in ["Tech", "Banks"]}
alloc.optimize()
print(f"Expected return: {100 * alloc.ObjVal:.2f}%")

var_report, con_report = sensitivity_report(alloc)
print(var_report.round(4))
print()
print(con_report.round(4))

* The **Tech limit** has shadow price $0.0877$, valid up to $40\%$: each extra percentage point allowed in Tech increases the expected return by $0.0877$ percentage points (we move money from THETA, 5.46%, to DELTA, 14.23%). This is the cost of the sector limit.
* The bond fund **IOTA** is not in the portfolio and has reduced cost $-0.0265$: its expected return (2.81%) must increase by 2.65 percentage points before it enters the portfolio (column `obj up`: 5.46%).
* Stocks at their 20% cap (e.g., ALFA) have a *positive* reduced cost: it is the shadow price of their upper bound, how much the expected return would increase per unit if their cap were raised.

### Exercise 7.1 ✏️ (Exercise 3 of Lecture 3)
What happens to the optimal expected return if the Tech limit is raised to 35%? And to 50%? Answer with the shadow price and its range, then check by solving the LP again.

In [ ]:
# YOUR CODE HERE

### Exercise 7.2 ✏️
By how much must the expected return of the gold fund KAPPA increase before it enters the optimal portfolio? Check your answer by changing its objective coefficient (`w["KAPPA"].Obj`) slightly above the threshold and solving again.

In [ ]:
# YOUR CODE HERE